# Equation finder: a tensor AI that writes the governing equation of its data

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/equation_finder)

Give it field data, get back the law in LaTeX. A library of candidate terms ($1, u, u^2, u_x, u u_x, u_{xx}, \nabla^2 E_z, \partial_x B_y, \dots$) is built with finite differences, Savitzky–Golay filters or the weak form, and sequentially thresholded ridge regression (PDE-FIND) keeps the few terms that explain $\partial_t u$. From simulated data it recovers the Lorenz system, the heat equation, viscous Burgers and Maxwell's equations from a Yee FDTD run (with $c$ to 0.2 %, 0.03 % with the weak form), and finds $\nabla\cdot\mathbf B = 0$ hidden in the library; a noise study shows where each derivative estimator breaks.

**Vector form**

$$\partial_t u = \Theta(u)\,\xi,\qquad \xi^\star = \operatorname*{arg\,min}_{\xi}\ \lVert \partial_t u - \Theta(u)\,\xi\rVert_2^2 + \lambda\lVert\xi\rVert_2^2 \ \ \text{with } |\hat\xi_k| < \tau \Rightarrow \xi_k = 0,\qquad -\!\int\! \partial_t\phi\, u = \sum_k \xi_k (-1)^{|\alpha_k|}\!\int\! (D^{\alpha_k}\phi)\, f_k(u)$$

**Index (tensor) form**

$$\Theta_{nk} = \sum_{\text{parts}} s\, D^{\alpha}\Big(\prod_g u_g^{p_g}\Big)_n \qquad \dot u_{nf} \approx \sum_j w^{(d)}_j\, u_{f,\,n+j} / h^d \qquad G_{pq,k} = \sum_{x,t} \Phi^{(\alpha_k)}_{px}\, \Psi_{qt}\, f_k(u)_{xt}\, h_x h_t \qquad \xi_{fk} = \Big[\big(\Theta^\top\Theta + \lambda I\big)^{-1}_{M_f}\, \Theta^\top \dot u_f\Big]_k$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\Theta_{nk}$ | `strong_library(U, h, terms, method)` | (rows, K) | the library: column $k$ is a candidate term $D^\alpha(\prod_g u_g^{p_g})$ at every interior grid point $n$, e.g. $u u_x = \tfrac12\partial_x(u^2)$, $\nabla^2 E_z = \partial_{xx}E_z + \partial_{yy}E_z$ |
| $w^{(d)}_j$ | `fd_kernel(d), sg_kernel(m, p, d)` | (2m+1,) | derivative engine: a 1D convolution along one axis; central differences, or Savitzky–Golay (least-squares polynomial of degree $p$ on $2m+1$ samples) after a cubic smoother |
| $G_{pq,k}$ | `contract(f, [Phi[a][d], Psi[0]])` | (P·Q, K) | weak form: $x$ and $t$ summed; the derivatives sit on the smooth bumps $\phi(s) = (1-s^2)^{10}$, never on the noisy data |
| $M_f$ | `masked_ridge(G, r, mask, lam)` | (E, K) | one batched ridge solve for every equation $f$ over its surviving terms; STRidge zeroes $/\hat\xi_k/ < \tau$ and refits |
| $\tau$ | `TOLS, select(path, A_val, b_val)` | (41,) | tolerance sweep: keep the model with the fewest terms whose validation error is within 5 % (+ $10^{-3}$) of the best |
| $\sigma(\alpha_{fk}/T)$ | `gate_search(A, b)` | (E, K) | the DARTS-style alternative: $\xi = \sigma(\alpha/T)\, w$ with a gate penalty, temperature annealed from 1 to 0.01 |
| $\pm$ | `einsum('bn,nk,nl->bkl', w, A, A)` | (B, k, k) | bootstrap: $B$ row resamplings as multinomial weights, all fits in one batched solve |
| $\hat\xi$ | `read_equation(write_equation(...), terms)` | (K,) | the written LaTeX read back by a parser: it reproduces the fitted model to the printed precision |

The idea. This is Jacky Ka Long Ko's "automatic physics-equation finder": a small tensor AI that looks at field data and writes down the equation that governs it, in LaTeX (the SINDy / PDE-FIND approach). It follows the operator search on MNIST, which ended with "a handwritten digit has no governing equation"; here the data come from laws, so there is something to find. The architecture has three modules: (1) a denoising smoother, (2) an einsum / finite-difference derivative engine, (3) sparse operator regression. Here (2) and (3) are complete and (1) is a Savitzky–Golay smoother (no neural network is needed for it). Method. Every candidate term is a derivative of a monomial of the fields, $D^\alpha(\prod_g u_g^{p_g})$: 20 monomials up to degree 3 for the Lorenz system, 13 terms $D^d(u^p)$ ($p, d \le 3$) for the 1D equations, 17 terms for Maxwell (the fields, their $x$ and $y$ derivatives, their Laplacians, a few products). Three ways to build the linear system $\partial_t u = \Theta\xi$: central finite differences (fd); Savitzky–Golay (sg: a cubic smoother on 7 samples, then a quartic derivative fit on 9), both 1D convolutions axis by axis; and the weak form (weak SINDy, Messenger and Bortz 2021), where the equation is integrated against bumps $(1-s^2)^{10}$ and integration by parts moves every derivative onto them, one tensor contraction per axis. STRidge (Rudy et al. 2017) then runs on unit-norm columns: ridge, zero every coefficient below $\tau$, refit, repeat; a sweep of 41 tolerances for 3 ridge strengths, and the model kept is the one with the fewest terms whose validation error is within 5 % (+ $10^{-3}$) of the best. The larger ridge strength matters for Maxwell: since $\nabla\cdot\mathbf B = 0$, the columns $\partial_x B_x$ and $\partial_y B_y$ are almost opposite, and with a weak ridge the regression added $2200\,\nabla\cdot\mathbf B$ to the $E_z$ equation. Coefficients are refitted by least squares on the kept terms, with a bootstrap over rows for the ±; that ± is statistical only and does not include the bias of the derivative estimate, so it is often far smaller than the true error. The alternative regression is the DARTS-style relaxation of operator_search_mnist: $\xi = \sigma(\alpha/T)\,w$ with a gate penalty and the temperature annealed from 1 to 0.01. Data, all simulated here: 8 Lorenz trajectories (RK4); the heat equation on a 10 cm periodic rod, exact by Fourier modes ($D$ = 0.2 cm²/s); viscous Burgers on $2\pi$ m, exact by the Cole–Hopf transform ($\nu$ = 0.1 m²/s); Maxwell in a periodic 6.4 cm box from a Yee FDTD run (TM mode, 0.5 mm cells, Courant number 0.5), sampled as a measurement would be (all fields interpolated to the same points and times). Noise: Gaussian, 0, 0.1, 1 and 5 % of each field's standard deviation. Results. The tests check that the kernels and the weak form are exact where calculus says so, that the data obey their laws (heat kernel, Cole–Hopf, Yee keeps $\nabla\cdot\mathbf B = 0$ to round-off and conserves its energy to 0.2 %), that STRidge recovers a planted sparse vector exactly, that every target is recovered from clean data and that the written LaTeX, read back by a parser, reproduces the fitted model. Bundled run, clean data, finite differences: Lorenz $\sigma$ = 9.9987, $\rho$ = 27.989, $\beta$ = 2.6662; heat $D$ = 0.20023 cm²/s; Burgers $u_t = -1.0007\,u u_x + 0.09996\,u_{xx}$; Maxwell $\partial_t E_z = c^2(\partial_x B_y - \partial_y B_x)$ with $c$ = 29.92 cm/ns (true 29.98) and $\partial_t \mathbf B = -\nabla\times\mathbf E$ with coefficients 0.996. The weak form is far more accurate (errors of $10^{-8}$ on the exact data; 0.03 % on Maxwell, which is the Yee scheme's own numerical dispersion, not the finder). The smallest singular vector of the Maxwell library is $0.707\,\partial_x B_x + 0.707\,\partial_y B_y$, with a singular value 220 times below the next one: the data reveal $\nabla\cdot\mathbf B = 0$ as an implicit law. Noise (right terms found?): finite differences fail from 0.1 % noise on heat and Burgers and from 1 % on Lorenz, and survive 5 % only on Maxwell (with 25 % coefficient error); Savitzky–Golay survives 0.1 % everywhere and 5 % only on Maxwell; the weak form finds the right terms at every level for heat, Burgers and Maxwell (at 5 % noise, coefficient errors 7 %, 6 % and 0.1 %) and up to 1 % for Lorenz (at 5 % it adds two tiny spurious terms to $\dot y$). The annealed gates match weak STRidge on Burgers and Maxwell but miss the $-y$ term of the Lorenz $\dot y$ at every noise level and for every penalty tried ($10^{-6}$ to $10^{-3}$): $25.5\,x - 0.95\,xz$ absorbs it, a local minimum the relaxation does not leave; they also fail on heat at 5 %. Re-simulating the equations found from the first snapshot reproduces the heat and Burgers data to $3\times 10^{-4}$ and Maxwell to 3 %; Lorenz is 3 % off after 5 time units because chaos amplifies a 0.01 % coefficient error. results/noise_study.csv and results/equations.tex hold every number and equation; one GIF per target (Maxwell is the main one) and results/summary.png. Verdict. The finder recovers known equation forms from clean, well-sampled data, and with the weak form it tolerates realistic noise; derivative estimation, not the regression, is what breaks first. It only finds terms that are in its library, it needs the right variables to be measured, and a discovered equation is a hypothesis: it does not discover new physics (or physiology) without independent validation. It is not a replacement for large language models either; it is a different task (fitting a law to data, not producing text). Related work. SINDy (Brunton, Proctor and Kutz 2016), PDE-FIND and STRidge (Rudy et al. 2017), weak SINDy (Messenger and Bortz 2021); symbolic regression by genetic programming (Schmidt and Lipson 2009) and AI Feynman (Udrescu and Tegmark 2020); the Yee scheme (Yee 1966). Try it. Fewer bootstrap resamplings for a quicker run, more gate steps to see whether the relaxation escapes its Lorenz minimum (it did not up to 4000), or a coarser Yee grid.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/equation_finder/main.py
"""Main loop: simulate the four targets, run the finder on every target × noise level × derivative estimator (plus the
annealed-gate alternative), look for an implicit law in the Maxwell library (∇·B = 0), re-simulate the equations found
from clean data, write the tables and the animation."""

import csv
import time
from pathlib import Path

from problems._common.runtime import cli, fast_mode

from . import equation as eq
from . import plot, solve
from . import tensors as T


def run(params: dict, out_dir: Path) -> Path:
    fast = fast_mode()
    t_all = time.perf_counter()
    datasets = {}
    for name in T.TARGETS:
        t0 = time.perf_counter()
        if name == "maxwell":
            datasets[name] = T.maxwell(n=64, steps=120) if fast else T.maxwell(n=params["maxwell_grid"])
        else:
            datasets[name] = T.dataset(name, small=fast)
        ds = datasets[name]
        print(f"{name:8s} data {tuple(ds['U'].shape)} (fields, axes…, time), library of {len(ds['terms'])} terms, "
              f"simulated in {time.perf_counter() - t0:.1f} s")  # fmt: skip

    levels = (0.0, 0.01) if fast else T.NOISE_LEVELS
    methods = ("fd", "weak") if fast else ("fd", "sg", "weak")

    def log(r):
        print(f"{r['target']:8s} {r['method']:9s} noise {r['noise'] * 100:4.1f} %: "
              f"{'right terms' if r['success'] else 'WRONG terms'}, largest coefficient error {r['coef_err']:.1e}, "
              f"{r['seconds']:.1f} s", flush=True)  # fmt: skip
        for line in r["lines"]:
            print("      ", line)

    rows = solve.noise_study(levels, methods, n_boot=10 if fast else params["bootstrap"], gate=not fast,
                             gate_steps=params["gate_steps"], datasets=datasets, log=log)  # fmt: skip

    ds = datasets["maxwell"]  # implicit law: the smallest singular vector of the (clean) Maxwell library
    th, _, _ = eq.strong_library(ds["U"], ds["h"], ds["terms"], "fd")
    vec, sv = solve.null_vector(th)
    vec = vec * vec.sign()[vec.abs().argmax()]
    law = solve.write_equation("0", ds["terms"], vec.where(vec.abs() > 0.05, 0.0), digits=3)
    print(f"implicit law hidden in the Maxwell library (smallest singular vector of the normalised Θ): {law}; "
          f"σ_min / σ_next = {sv[0] / sv[1]:.1e}")  # fmt: skip

    show = {n: next(r for r in rows if r["target"] == n and r["method"] == "fd" and r["noise"] == 0) for n in datasets}
    sims = {}
    for n, r in show.items():
        sims[n] = solve.simulate_found(datasets[n], r["xi"])
        U = datasets[n]["U"]
        rel = float((sims[n][..., -1] - U[..., -1]).norm() / U[..., -1].norm())
        print(f"{n:8s} clean data, finite differences: {plot.constants(n, r['terms'], r['xi']).replace('$', '')}; "
              f"re-simulated from the first snapshot, off the data by {rel:.1e} (relative) at the end")  # fmt: skip

    res = out_dir / "results"
    res.mkdir(parents=True, exist_ok=True)
    with open(res / "noise_study.csv", "w", newline="", encoding="utf-8") as f:
        f.write("# equation_finder: one row per (target, regression/derivative method, noise level). noise = std "
                "of the added Gaussian noise / std of the field; coef_error = largest |xi - xi_true| / |xi_true| "
                "over the true terms (1 = a true term was missed)\n")  # fmt: skip
        w = csv.writer(f)
        w.writerow(
            ["target", "method", "noise", "right_terms", "coef_error", "n_terms", "rows", "seconds", "equations"]
        )
        for r in rows:
            w.writerow([r["target"], r["method"], r["noise"], int(r["success"]), f"{r['coef_err']:.3e}",
                        int(r["support"].sum()), r["rows"], f"{r['seconds']:.2f}", " ; ".join(r["lines"])])  # fmt: skip
    with open(res / "equations.tex", "w", encoding="utf-8") as f:
        f.write("% Equations found by equation_finder (STRidge or annealed gates on a library of candidate terms).\n"
                "% fd = finite differences, sg = Savitzky-Golay, weak = weak form; +- = bootstrap standard deviation\n"
                "% over rows (statistical only: no bias of the derivative estimate).\n")  # fmt: skip
        for r in rows:
            f.write(f"% {r['target']}, {r['method']}, noise {r['noise'] * 100:g} %: "
                    f"{'right terms' if r['success'] else 'wrong terms'}\n")  # fmt: skip
            f.write("\\begin{aligned}\n" + " \\\\\n".join(ln.replace(" = ", " &= ", 1) for ln in r["lines"]))
            f.write("\n\\end{aligned}\n\n")
        f.write(
            f"% implicit law from the null space of the Maxwell library (sigma_min/sigma_next = {sv[0] / sv[1]:.1e}):\n"
        )
        f.write(law + "\n")
    print(f"study {time.perf_counter() - t_all:.1f} s; animation next{' (fast mode)' if fast else ''}")
    return plot.animate(datasets, rows, show, sims, out_dir)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/equation_finder/equation.py
r"""Physics → tensor translation: the library $\Theta(u)$ of candidate terms and the derivative engine.

The finder looks for a sparse $\xi$ in

    \partial_t u_f = \sum_k \Theta_k(u)\, \xi_{fk}

where every candidate term is a derivative of a monomial of the fields, $\Theta_k = \sum_{\text{parts}} s\, D^\alpha
\big(\prod_g u_g^{p_g}\big)$ (for example $u u_x = \tfrac12 \partial_x(u^2)$, $\nabla^2 E_z = \partial_{xx}E_z +
\partial_{yy}E_z$). Fields are tensors U of shape (F, *axes) whose last axis is time.

Two ways to turn data into the linear system:

* strong form (the PDE-FIND way): estimate every derivative on the grid, either with plain central finite differences
  (fd) or with Savitzky–Golay filters (sg: smooth the fields with a local cubic fit, then differentiate a local
  quartic fit). Both are 1D convolutions applied axis by axis.
* weak form (weak SINDy): multiply by smooth bumps $\phi_p(x)\psi_q(t)$ and integrate by parts, so all derivatives
  fall on the known test functions and none is taken of the noisy data:

    -\int \partial_t\phi\, u = \sum_k \xi_k\, (-1)^{|\alpha_k|} \int (D^{\alpha_k}\phi)\, f_k(u)

  Each integral is one tensor contraction per axis: G_{pq} = \sum_{x,t} \Phi_{px}\, \Psi_{qt}\, f(u)_{xt}\, h_x h_t
  (x and t summed).
"""

import itertools
import math
from typing import NamedTuple

import numpy as np
import torch
import torch.nn.functional as F


class Term(NamedTuple):
    latex: str  # how the term is written (and read back)
    parts: tuple  # ((scale, powers per field, derivative order per spatial axis), ...): Θ = Σ scale · D^α(Π u^p)


def _t(latex: str, *parts) -> Term:
    return Term(latex, tuple(parts))


# ---------------------------------------------------------------- the libraries
SCALAR_1D = (  # u(x, t): D^d(u^p) for p ≤ 3, d ≤ 3  (13 terms)
    _t("1", (1.0, (0,), (0,))),
    _t("u", (1.0, (1,), (0,))),
    _t("u^2", (1.0, (2,), (0,))),
    _t("u^3", (1.0, (3,), (0,))),
    _t("u_{x}", (1.0, (1,), (1,))),
    _t("u u_{x}", (0.5, (2,), (1,))),  # ½ ∂_x(u²)
    _t("u^2 u_{x}", (1 / 3, (3,), (1,))),  # ⅓ ∂_x(u³)
    _t("u_{xx}", (1.0, (1,), (2,))),
    _t("\\partial_{xx}(u^2)", (1.0, (2,), (2,))),
    _t("\\partial_{xx}(u^3)", (1.0, (3,), (2,))),
    _t("u_{xxx}", (1.0, (1,), (3,))),
    _t("\\partial_{xxx}(u^2)", (1.0, (2,), (3,))),
    _t("\\partial_{xxx}(u^3)", (1.0, (3,), (3,))),
)


def _monomial_latex(powers, names) -> str:
    out = [n if p == 1 else f"{n}^{p}" for n, p in zip(names, powers) if p > 0]
    return " ".join(out) if out else "1"


LORENZ = tuple(  # all monomials of x, y, z up to degree 3 (20 terms), no spatial axis
    _t(_monomial_latex(p, "xyz"), (1.0, p, ()))
    for deg in range(4)
    for p in sorted((q for q in itertools.product(range(deg + 1), repeat=3) if sum(q) == deg), reverse=True)
)


def _maxwell() -> tuple:
    fields = ("E_z", "B_x", "B_y")
    one = [(1, 0, 0), (0, 1, 0), (0, 0, 1)]
    out = [_t("1", (1.0, (0, 0, 0), (0, 0)))] + [_t(f, (1.0, p, (0, 0))) for f, p in zip(fields, one)]
    out += [
        _t(f"\\partial_{a} {f}", (1.0, p, d)) for f, p in zip(fields, one) for a, d in (("x", (1, 0)), ("y", (0, 1)))
    ]
    out += [_t(f"\\nabla^2 {f}", (1.0, p, (2, 0)), (1.0, p, (0, 2))) for f, p in zip(fields, one)]
    out += [_t("E_z^2", (1.0, (2, 0, 0), (0, 0))), _t("E_z^3", (1.0, (3, 0, 0), (0, 0))),
            _t("E_z B_x", (1.0, (1, 1, 0), (0, 0))), _t("E_z B_y", (1.0, (1, 0, 1), (0, 0)))]  # fmt: skip
    return tuple(out)


MAXWELL = _maxwell()  # fields (E_z, B_x, B_y) on (x, y): 17 terms


def truth_matrix(terms, rows: list[dict]) -> torch.Tensor:
    """The true ξ as a (F, K) tensor from one {term latex: coefficient} dict per equation."""
    names = [tm.latex for tm in terms]
    xi = torch.zeros(len(rows), len(terms), dtype=torch.float64)
    for f, row in enumerate(rows):
        for k, v in row.items():
            xi[f, names.index(k)] = v
    return xi


# ---------------------------------------------------------------- derivative engine (module 2) and smoother (module 1)
SG_SMOOTH = (3, 3)  # Savitzky–Golay smoother: half-width 3 (7 samples), cubic fit
SG_DERIV = (4, 4)  # Savitzky–Golay derivative: half-width 4 (9 samples), quartic fit


def sg_kernel(m: int, p: int, d: int) -> np.ndarray:
    """Weights w_j (j = −m..m) with f^{(d)}(0) ≈ Σ_j w_j f(j) for a least-squares polynomial of degree p on 2m + 1
    samples (Savitzky and Golay 1964). Exact for polynomials of degree ≤ p."""
    j = np.arange(-m, m + 1, dtype=float)
    V = j[:, None] ** np.arange(p + 1)  # (2m+1, p+1) Vandermonde
    return math.factorial(d) * np.linalg.pinv(V)[d]


def fd_kernel(d: int) -> np.ndarray:
    """The narrowest central finite difference for the d-th derivative: polynomial interpolation through 2m+1 points."""
    m = max(1, (d + 1) // 2)
    return sg_kernel(m, 2 * m, d)


def filter_axis(f: torch.Tensor, w: np.ndarray, axis: int, scale: float = 1.0) -> torch.Tensor:
    """Cross-correlate f with the kernel w along one axis (zero padding; the edges are cropped later)."""
    m = len(w) // 2
    g = f.movedim(axis, -1)
    shape = g.shape
    k = torch.as_tensor(w, dtype=f.dtype, device=f.device).view(1, 1, -1)
    out = F.conv1d(g.reshape(-1, 1, shape[-1]), k, padding=m).reshape(shape)
    return out.movedim(-1, axis) / scale


def derivative(f: torch.Tensor, axis: int, d: int, h: float, method: str) -> torch.Tensor:
    if d == 0:
        return f
    w = fd_kernel(d) if method == "fd" else sg_kernel(SG_DERIV[0], max(SG_DERIV[1], d), d)
    return filter_axis(f, w, axis, h**d)


def margin(method: str) -> int:
    return 2 if method == "fd" else SG_SMOOTH[0] + SG_DERIV[0]


def monomial(U: torch.Tensor, powers) -> torch.Tensor:
    """Π_g U_g^{p_g}.   U: (F, *axes) → (*axes)"""
    out = torch.ones_like(U[0])
    for g, p in enumerate(powers):
        if p:
            out = out * U[g] ** p
    return out


def spatial_axes(h) -> list[int]:
    """Axes that carry a spatial derivative: every axis but the last (time) and batch axes (h = None)."""
    return [a for a in range(len(h) - 1) if h[a] is not None]


def strong_library(U: torch.Tensor, h, terms, method: str = "fd"):
    """Θ (rows, K) and ∂_t U (rows, F) at every interior grid point; crop = the slices that pick the interior.

    U: (F, *axes), h: spacing per axis (None for a batch axis), method: 'fd' or 'sg'."""
    h = [None if v is None else float(v) for v in h]
    sp, ta = spatial_axes(h), len(h) - 1
    if method == "sg":  # module 1: the smoother (local cubic fit along every axis)
        w = sg_kernel(SG_SMOOTH[0], SG_SMOOTH[1], 0)
        for a in sp + [ta]:
            U = filter_axis(U, w, a + 1)
    m = margin(method)
    crop = tuple(slice(m, -m) if (h[a] is not None) else slice(None) for a in range(len(h)))
    cols = []
    for tm in terms:
        col = 0
        for s, powers, alpha in tm.parts:
            f = monomial(U, powers)
            for a, d in zip(sp, alpha):
                f = derivative(f, a, d, h[a], method)
            col = col + s * f
        cols.append((col * torch.ones_like(U[0]))[crop].reshape(-1))
    ut = torch.stack([derivative(U[g], ta, 1, h[ta], method)[crop].reshape(-1) for g in range(len(U))], 1)
    return torch.stack(cols, 1), ut, crop


# ---------------------------------------------------------------- weak form
BUMP_POWER = 10  # φ(s) = (1 − s²)^10 on |s| ≤ 1: it and its first 9 derivatives vanish at the edge


def bump_matrices(n: int, h: float, radius: int, stride: int, orders: int = 4) -> list[torch.Tensor]:
    """[Φ^(0), …, Φ^(orders−1)]: Φ^(d)_{pn} = d^dφ/dx^d of the bump centred at c_p, radius r samples, at sample n."""
    centres = np.arange(radius, n - radius, stride)
    s = (np.arange(n)[None, :] - centres[:, None]) / radius  # (P, n)
    poly = np.polynomial.Polynomial([1.0, 0.0, -1.0]) ** BUMP_POWER
    out = []
    for d in range(orders):
        vals = poly.deriv(d)(s) if d else poly(s)
        vals = np.where(np.abs(s) <= 1, vals, 0.0) / (radius * h) ** d
        out.append(torch.tensor(vals))
    return out


def test_functions(shape, h, radius, stride) -> list:
    """Per data axis (shape = U.shape[1:]): the bump matrices, None on a batch axis. radius, stride in samples."""
    return [None if h[a] is None else bump_matrices(shape[a], float(h[a]), radius[a], stride[a]) for a in range(len(h))]


def contract(f: torch.Tensor, mats: list) -> torch.Tensor:
    """f contracted with one matrix per axis (None = keep that axis): G_{pq…} = Σ_{xt…} M1_{px} M2_{qt} … f_{xt…}."""
    for a, M in enumerate(mats):
        if M is not None:
            f = torch.tensordot(M.to(f), f, dims=([1], [a])).movedim(0, a)
    return f


def weak_library(U: torch.Tensor, h, terms, phi):
    """Weak-form Θ (rows, K) and target (rows, F): one row per test function (and batch member)."""
    h = [None if v is None else float(v) for v in h]
    sp, ta = spatial_axes(h), len(h) - 1
    vol = math.prod(v for v in h if v is not None)
    cols = []
    for tm in terms:
        col = 0
        for s, powers, alpha in tm.parts:
            mats = [None] * len(h)
            for a, d in zip(sp, alpha):
                mats[a] = phi[a][d]
            for a in sp:
                mats[a] = mats[a] if mats[a] is not None else phi[a][0]
            mats[ta] = phi[ta][0]
            f = monomial(U, powers) * torch.ones_like(U[0])
            col = col + s * (-1) ** sum(alpha) * contract(f, mats) * vol
        cols.append(col.reshape(-1))
    mats = [phi[a][0] if a in sp else None for a in range(len(h))]
    mats[ta] = phi[ta][1]
    ut = torch.stack([-contract(U[g], mats).reshape(-1) * vol for g in range(len(U))], 1)
    return torch.stack(cols, 1), ut


# ---------------------------------------------------------------- the found equation as a simulator
def spectral_derivative(f: torch.Tensor, axis: int, d: int, h: float) -> torch.Tensor:
    """d-th derivative along a periodic axis by Fourier modes: multiply by (ik)^d."""
    if d == 0:
        return f
    n = f.shape[axis]
    k = 2 * math.pi * torch.fft.fftfreq(n, d=h, dtype=f.dtype)
    if d % 2 and n % 2 == 0:
        k[n // 2] = 0  # odd derivatives of the Nyquist mode are not defined: drop it
    shape = [1] * f.dim()
    shape[axis] = n
    return torch.fft.ifft(torch.fft.fft(f, dim=axis) * (1j * k.view(shape)) ** d, dim=axis).real


def found_rhs(U: torch.Tensor, h_space, terms, xi: torch.Tensor) -> torch.Tensor:
    """∂_t U_f = Σ_k ξ_fk Θ_k(U) for one time slice, spectral derivatives (periodic space). (F, *space) → (F, *space).
    Only the terms some equation uses are evaluated."""
    sp = [a for a in range(len(h_space)) if h_space[a] is not None]
    if not sp:  # an ODE: all monomials at once, Θ_k = Π_g U_g^{P_gk}, then ξ_fk Θ_k (k summed)
        P = torch.tensor([tm.parts[0][1] for tm in terms], dtype=U.dtype).T  # (F, K)
        theta = torch.prod(U.unsqueeze(1) ** P.view(*P.shape, *[1] * (U.dim() - 1)), dim=0)  # (K, *rest)
        return torch.einsum("fk,k...->f...", xi, theta)
    out = torch.zeros_like(U)
    for k in torch.nonzero(xi.abs().sum(0)).flatten().tolist():
        col = 0
        for s, powers, alpha in terms[k].parts:
            f = monomial(U, powers) * torch.ones_like(U[0])
            for a, d in zip(sp, alpha):
                f = spectral_derivative(f, a, d, float(h_space[a]))
            col = col + s * f
        out = out + xi[:, k].view(-1, *[1] * (U.dim() - 1)) * col
    return out


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/equation_finder/tensors.py
"""Concrete tensors: the four data sets, all simulated here (nothing downloaded), and the noise model.

Every data set is a dict with U (F, *axes) whose last axis is time, the spacing h per axis (None for a batch axis of
independent trajectories), coordinates, the library to search, the true ξ (F, K) and the weak-form test functions.

1. Lorenz system (ODE): 8 trajectories from random starts, RK4.
2. Heat equation u_t = D u_xx on a periodic rod: exact, by Fourier modes decaying as e^{-D k² t}.
3. Viscous Burgers u_t = −u u_x + ν u_xx: exact, by the Cole–Hopf transform u = −2ν φ_x / φ of a heat solution φ.
4. Maxwell's equations in 2D vacuum (TM mode E_z, B_x, B_y): a Yee FDTD simulation (Yee 1966), sampled as a
   measurement would be (all fields interpolated to the same points and times).
"""

import math

import torch

from . import equation as eq

DT = torch.float64  # every tensor in float64: derivatives of data amplify round-off

TARGETS = ("lorenz", "heat", "burgers", "maxwell")
LORENZ_PARAMS = (10.0, 28.0, 8.0 / 3.0)  # σ, ρ, β
HEAT_D = 0.2  # thermal diffusivity (cm²/s)
BURGERS_NU = 0.1  # viscosity (m²/s)
C_LIGHT = 29.9792458  # speed of light (cm/ns)
NOISE_LEVELS = (0.0, 0.001, 0.01, 0.05)  # Gaussian noise, fraction of each field's standard deviation


# ---------------------------------------------------------------- 1. Lorenz
def lorenz_rhs(X: torch.Tensor) -> torch.Tensor:
    """(ẋ, ẏ, ż) = (σ(y − x), x(ρ − z) − y, xy − βz).   X: (..., 3)"""
    s, r, b = LORENZ_PARAMS
    x, y, z = X.unbind(-1)
    return torch.stack([s * (y - x), x * (r - z) - y, x * y - b * z], -1)


def lorenz(n_traj: int = 8, t_end: float = 5.0, dt: float = 0.0025, substeps: int = 2, seed: int = 0) -> dict:
    g = torch.Generator().manual_seed(seed)
    X = torch.rand(n_traj, 3, generator=g, dtype=DT) * torch.tensor([30.0, 40.0, 35.0], dtype=DT) - torch.tensor(
        [15.0, 20.0, -5.0], dtype=DT
    )
    n_t, k = round(t_end / dt) + 1, dt / substeps
    out = [X]
    for _ in range((n_t - 1) * substeps):  # RK4, all trajectories at once
        k1 = lorenz_rhs(X)
        k2 = lorenz_rhs(X + 0.5 * k * k1)
        k3 = lorenz_rhs(X + 0.5 * k * k2)
        X = X + k / 6 * (k1 + 2 * k2 + 2 * k3 + lorenz_rhs(X + k * k3))
        out.append(X)
    U = torch.stack(out[::substeps], -1).permute(1, 0, 2)  # (3, traj, t)
    s, r, b = LORENZ_PARAMS
    truth = eq.truth_matrix(eq.LORENZ, [{"x": -s, "y": s}, {"x": r, "y": -1.0, "x z": -1.0}, {"x y": 1.0, "z": -b}])
    return {"name": "lorenz", "U": U, "h": (None, dt), "coords": [torch.arange(n_traj, dtype=DT),
            torch.arange(n_t, dtype=DT) * dt],
            "fields": ["x", "y", "z"], "lhs": ["\\dot{x}", "\\dot{y}", "\\dot{z}"], "terms": eq.LORENZ,
            "truth": truth, "weak": {"radius": (None, 60), "stride": (None, 30)}}  # fmt: skip


# ---------------------------------------------------------------- 2. heat
def heat(n_x: int = 256, L: float = 10.0, t_end: float = 5.0, n_t: int = 101, bumps=None, seed: int = 0) -> dict:
    """Temperature excess u (K) on a periodic rod of length L (cm); initial Gaussian bumps (centre, width, height)."""
    g = torch.Generator().manual_seed(seed)
    if bumps is None:
        r = torch.rand(4, 3, generator=g, dtype=DT)
        bumps = [(L * a, 0.3 + 0.5 * b, 1.0 + 2.0 * c) for a, b, c in r.tolist()]
    x, t = torch.arange(n_x, dtype=DT) * L / n_x, torch.linspace(0, t_end, n_t, dtype=DT)
    u0 = sum(A * torch.exp(-(((x - c + L / 2) % L - L / 2) ** 2) / (2 * s**2)) for c, s, A in bumps)
    k = 2 * math.pi * torch.fft.rfftfreq(n_x, d=L / n_x, dtype=DT)
    u = torch.fft.irfft(torch.fft.rfft(u0)[:, None] * torch.exp(-HEAT_D * k[:, None] ** 2 * t[None, :]), n=n_x, dim=0)
    return {"name": "heat", "U": u[None], "h": (L / n_x, t_end / (n_t - 1)), "coords": [x, t], "fields": ["u"],
            "lhs": ["u_t"], "terms": eq.SCALAR_1D, "truth": eq.truth_matrix(eq.SCALAR_1D, [{"u_{xx}": HEAT_D}]),
            "weak": {"radius": (16, 10), "stride": (8, 5)}, "bumps": bumps}  # fmt: skip


# ---------------------------------------------------------------- 3. Burgers
def burgers(n_x: int = 256, t_end: float = 3.0, n_t: int = 151) -> dict:
    """Velocity u (m/s) on a periodic line of length 2π m. Cole–Hopf: u0 = −2ν ∂_x log φ0, φ0 = exp(−∫u0 / 2ν), then φ
    solves the heat equation φ_t = ν φ_xx exactly (Fourier modes), and u = −2ν φ_x / φ solves Burgers exactly."""
    L = 2 * math.pi
    x, t = torch.arange(n_x, dtype=DT) * L / n_x, torch.linspace(0, t_end, n_t, dtype=DT)
    U0 = 0.6 * (1 - torch.cos(x)) + 0.15 * (
        math.cos(1.0) - torch.cos(2 * x + 1)
    )  # ∫ u0, u0 = 0.6 sin x + 0.3 sin(2x+1)
    phi0 = torch.exp(-U0 / (2 * BURGERS_NU))
    k = 2 * math.pi * torch.fft.rfftfreq(n_x, d=L / n_x, dtype=DT)
    ph = torch.fft.rfft(phi0)[:, None] * torch.exp(-BURGERS_NU * k[:, None] ** 2 * t[None, :])  # (k, t)
    phi = torch.fft.irfft(ph, n=n_x, dim=0)
    phi_x = torch.fft.irfft(1j * k[:, None] * ph, n=n_x, dim=0)
    u = -2 * BURGERS_NU * phi_x / phi
    truth = eq.truth_matrix(eq.SCALAR_1D, [{"u u_{x}": -1.0, "u_{xx}": BURGERS_NU}])
    return {"name": "burgers", "U": u[None], "h": (L / n_x, t_end / (n_t - 1)), "coords": [x, t], "fields": ["u"],
            "lhs": ["u_t"], "terms": eq.SCALAR_1D, "truth": truth,
            "weak": {"radius": (16, 10), "stride": (8, 6)}}  # fmt: skip


# ---------------------------------------------------------------- 4. Maxwell (Yee FDTD)
def yee_fdtd(n: int = 128, h: float = 0.05, courant: float = 0.5, steps: int = 240, stride: int = 3, start: int = 6,
             seed: int = 0, record: bool = False) -> dict:  # fmt: skip
    """TM mode in a periodic square box (side n h cm): E_z at cell corners (i, j) and integer steps, B_x at (i, j+½),
    B_y at (i+½, j) and half steps (Yee 1966). Leap-frog:
        B_x ← B_x − Δt ∂_y E_z,   B_y ← B_y + Δt ∂_x E_z,   E_z ← E_z + c² Δt (∂_x B_y − ∂_y B_x)
    Snapshots every `stride` steps, with B interpolated (4-point cubic, in space and time) to the E_z points and times.
    Courant number S = c Δt / h must stay ≤ 1/√2 in 2D."""
    if courant > 1 / math.sqrt(2):
        raise ValueError(f"Courant number {courant} > 1/√2: the 2D Yee scheme would be unstable")
    c, dt, L = C_LIGHT, courant * h / C_LIGHT, n * h
    g = torch.Generator().manual_seed(seed)
    x = torch.arange(n, dtype=DT) * h
    E = torch.zeros(n, n, dtype=DT)
    for cx, cy, s, A in (
        torch.rand(3, 4, generator=g, dtype=DT) * torch.tensor([L, L, 0.3, 1.5], dtype=DT)
        + torch.tensor([0, 0, 0.5, 0.5], dtype=DT)
    ).tolist():
        dx, dy = (x[:, None] - cx + L / 2) % L - L / 2, (x[None, :] - cy + L / 2) % L - L / 2
        E = E + A * torch.exp(-(dx**2 + dy**2) / (2 * s**2))
    Bx, By = torch.zeros(n, n, dtype=DT), torch.zeros(n, n, dtype=DT)  # B^{-1/2} = 0: the pulses start at rest
    Es, Bxs, Bys, energy, div_b = [E], [], [], [], []
    for _ in range(steps):
        Bx_new = Bx - dt * (torch.roll(E, -1, 1) - E) / h  # ∂_y E_z at (i, j+½)
        By_new = By + dt * (torch.roll(E, -1, 0) - E) / h  # ∂_x E_z at (i+½, j)
        if record:  # Yee's conserved energy: E^n·E^n + c² B^{n−½}·B^{n+½}
            energy.append(float((E * E).sum() + c**2 * (Bx * Bx_new + By * By_new).sum()) * h * h)
        Bx, By = Bx_new, By_new
        curl = (By - torch.roll(By, 1, 0)) / h - (Bx - torch.roll(Bx, 1, 1)) / h  # at (i, j)
        E = E + c**2 * dt * curl
        Es.append(E), Bxs.append(Bx), Bys.append(By)
        if record:
            div_b.append(((torch.roll(Bx, -1, 0) - Bx) / h + (torch.roll(By, -1, 1) - By) / h).abs().max())

    def cubic(a, b, cc, d):  # value midway between b and cc from 4 equally spaced samples
        return (-a + 9 * b + 9 * cc - d) / 16

    snaps = list(range(start, steps - 1, stride))
    Ez = torch.stack([Es[k] for k in snaps], -1)
    Bx_t = torch.stack([cubic(Bxs[k - 2], Bxs[k - 1], Bxs[k], Bxs[k + 1]) for k in snaps], -1)  # B^{k±½, k±3/2} → B^k
    By_t = torch.stack([cubic(Bys[k - 2], Bys[k - 1], Bys[k], Bys[k + 1]) for k in snaps], -1)
    Bx_c = cubic(torch.roll(Bx_t, 2, 1), torch.roll(Bx_t, 1, 1), Bx_t, torch.roll(Bx_t, -1, 1))  # (i, j+½) → (i, j)
    By_c = cubic(torch.roll(By_t, 2, 0), torch.roll(By_t, 1, 0), By_t, torch.roll(By_t, -1, 0))  # (i+½, j) → (i, j)
    out = {"U": torch.stack([Ez, Bx_c, By_c]), "h": (h, h, stride * dt), "x": x,
           "t": torch.tensor(snaps, dtype=DT) * dt, "dt": dt}  # fmt: skip
    if record:
        out |= {"energy": torch.tensor(energy), "div_b": torch.stack(div_b),
                "b_scale": max(float(b.abs().max()) for b in Bxs + Bys) / h}  # fmt: skip
    return out


def maxwell(n: int = 128, steps: int = 240) -> dict:
    sim = yee_fdtd(n=n, steps=steps)
    c2 = C_LIGHT**2
    truth = eq.truth_matrix(eq.MAXWELL, [{"\\partial_x B_y": c2, "\\partial_y B_x": -c2}, {"\\partial_y E_z": -1.0},
                                         {"\\partial_x E_z": 1.0}])  # fmt: skip
    return {"name": "maxwell", "U": sim["U"], "h": sim["h"], "coords": [sim["x"], sim["x"], sim["t"]],
            "fields": ["E_z", "B_x", "B_y"], "lhs": ["\\partial_t E_z", "\\partial_t B_x", "\\partial_t B_y"],
            "terms": eq.MAXWELL, "truth": truth, "weak": {"radius": (10, 10, 8), "stride": (8, 8, 6)}}  # fmt: skip


def dataset(name: str, small: bool = False) -> dict:
    """One target. small=True: a lighter version for the tests."""
    if name == "lorenz":
        return lorenz(n_traj=4 if small else 8)
    if name == "heat":
        return heat()
    if name == "burgers":
        return burgers()
    if name == "maxwell":
        return maxwell(n=96, steps=150) if small else maxwell()
    raise ValueError(name)


def add_noise(U: torch.Tensor, level: float, seed: int) -> torch.Tensor:
    """Gaussian noise of standard deviation level × std(U_f), field by field."""
    if level == 0:
        return U
    g = torch.Generator().manual_seed(seed)
    std = U.reshape(len(U), -1).std(1).view(-1, *[1] * (U.dim() - 1))
    return U + level * std * torch.randn(U.shape, generator=g, dtype=U.dtype)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/equation_finder/solve.py
r"""The finder (module 3): sparse regression of $\partial_t u$ on the library, its uncertainty, the annealed-gate
alternative, and the LaTeX writer with its reader.

STRidge (Rudy et al. 2017): with unit-norm columns and target, solve a ridge regression, zero every coefficient below
a tolerance, refit on the survivors, repeat. A sweep of tolerances gives a path from the full library to the empty
one. Paths are run for three ridge strengths $\lambda$; the model kept is the one with the fewest terms whose
validation error stays within 5 % (+ 10⁻³) of the best model found, ties broken by the lower validation error.
"""

import math
import re
import time

import torch

from . import equation as eq
from . import tensors as T

TOLS = torch.logspace(-4, 0, 41, dtype=torch.float64)  # STRidge tolerances (normalised coefficients)
LAMS = (1e-8, 1e-5, 1e-3)  # ridge strengths on unit-norm columns, one path each (collinear libraries need the larger)
LAM = 1e-5
SLACK, FLOOR = 0.05, 1e-3  # model selection: sparsest with err ≤ (1 + SLACK) min err + FLOOR
MAX_ROWS = 20_000  # strong-form rows kept (random subset) for the regression


# ---------------------------------------------------------------- STRidge
def _normalise(A: torch.Tensor, b: torch.Tensor):
    sa, sb = A.norm(dim=0), b.norm(dim=0)
    sa, sb = torch.where(sa > 0, sa, 1.0), torch.where(sb > 0, sb, 1.0)
    return A / sa, b / sb, sa, sb


def masked_ridge(G: torch.Tensor, r: torch.Tensor, mask: torch.Tensor, lam: float) -> torch.Tensor:
    """For every equation e at once: argmin ‖A ξ_e − b_e‖² + λ‖ξ_e‖² over the columns in mask_e (others 0).
    G = AᵀA (K, K), r = (Aᵀb)ᵀ (E, K), mask (E, K) → ξ (E, K)."""
    m = mask.to(G.dtype)
    Gm = G[None] * m[:, :, None] * m[:, None, :] + lam * torch.eye(len(G), dtype=G.dtype)
    return torch.linalg.solve(Gm, (r * m).unsqueeze(-1)).squeeze(-1) * m


def _stridge_normalised(G, r, tol: float, lam: float, iters: int = 10) -> torch.Tensor:
    mask = torch.ones_like(r, dtype=torch.bool)
    for _ in range(iters):
        xi = masked_ridge(G, r, mask, lam)
        new = mask & (xi.abs() >= tol)
        if torch.equal(new, mask):
            break
        mask = new
    return masked_ridge(G, r, mask, 1e-12)  # unbiased refit on the support


def stridge(A: torch.Tensor, b: torch.Tensor, tol: float, lam: float = LAM) -> torch.Tensor:
    """ξ (E, K) in physical units for A (n, K), b (n, E)."""
    An, bn, sa, sb = _normalise(A, b)
    xi = _stridge_normalised(An.T @ An, (An.T @ bn).T, tol, lam)
    return xi * sb[:, None] / sa[None, :]


def stridge_path(A, b, tols=TOLS, lam: float = LAM) -> torch.Tensor:
    """ξ along the tolerance sweep: (len(tols), E, K), physical units."""
    An, bn, sa, sb = _normalise(A, b)
    G, r = An.T @ An, (An.T @ bn).T
    return torch.stack([_stridge_normalised(G, r, float(t), lam) for t in tols]) * sb[None, :, None] / sa[None, None, :]


def select(path: torch.Tensor, A_val, b_val) -> tuple[torch.Tensor, torch.Tensor]:
    """Per equation, the index of the model on the path (M, E, K) with the fewest terms among those whose validation
    error is within (1 + SLACK) of the best (+ FLOOR), ties go to the lower error. Returns index (E,), err (M, E)."""
    err = (torch.einsum("nk,mek->mne", A_val, path) - b_val[None]).norm(dim=1) / b_val.norm(dim=0)  # (M, E)
    ok = err <= (1 + SLACK) * err.min(0).values + FLOOR
    nnz = (path != 0).sum(-1).to(err.dtype)  # (M, E)
    score = torch.where(ok, nnz, torch.inf) + err.clamp(max=0.999)  # fewest terms first, then the smaller error
    return score.argmin(0), err


def bootstrap(A: torch.Tensor, b: torch.Tensor, n_boot: int = 100, seed: int = 0):
    """Least-squares coefficients on the given columns and their spread over row resamplings (with replacement).
    The resamples are row weights w_{βn} (multinomial counts), so all n_boot fits are one batched solve:
        G_{βkl} = Σ_n w_{βn} A_{nk} A_{nl},   r_{βk} = Σ_n w_{βn} A_{nk} b_n."""
    if A.shape[1] == 0:
        return torch.zeros(0, dtype=A.dtype), torch.zeros(0, dtype=A.dtype)
    sa = A.norm(dim=0)
    An = A / sa
    est = torch.linalg.lstsq(An, b[:, None]).solution[:, 0] / sa
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(len(b), (n_boot, len(b)), generator=g)
    w = torch.zeros(n_boot, len(b), dtype=A.dtype).scatter_add_(1, idx, torch.ones(idx.shape, dtype=A.dtype))
    G = torch.einsum("bn,nk,nl->bkl", w, An, An)
    r = torch.einsum("bn,nk,n->bk", w, An, b)
    boots = torch.linalg.solve(G, r.unsqueeze(-1)).squeeze(-1) / sa
    return est, boots.std(0)


def null_vector(A: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """The unit vector v minimising ‖Â v‖ over the column-normalised library Â (an implicit law Σ v_k Θ_k = 0), and the
    singular values in ascending order."""
    An = A / torch.where(A.norm(dim=0) > 0, A.norm(dim=0), 1.0)
    ev, V = torch.linalg.eigh(An.T @ An)
    return V[:, 0], ev.clamp(min=0).sqrt()


# ---------------------------------------------------------------- the DARTS-style alternative: annealed gates
def gate_search(A, b, steps: int = 1500, lam: float = 1e-3, temp=(1.0, 0.01), lr: float = 0.05, seed: int = 0,
                max_rows: int = 5000) -> dict:  # fmt: skip
    """A continuous relaxation of the support: ξ_ek = σ(α_ek / T) w_ek, trained with Adam on
        L = Σ_e ‖Â ξ_e − b̂_e‖² + λ Σ_ek σ(α_ek / T)
    while the temperature T falls geometrically (as in operator_search_mnist), so every gate ends open or shut. The
    support is the open gates; the coefficients are then refitted by least squares."""
    g = torch.Generator().manual_seed(seed)
    rows = torch.randperm(len(A), generator=g)[:max_rows]
    An, bn, sa, sb = _normalise(A[rows], b[rows])
    w = masked_ridge(An.T @ An, (An.T @ bn).T, torch.ones(b.shape[1], A.shape[1], dtype=torch.bool), LAM)
    w = w.clone().requires_grad_(True)
    alpha = torch.zeros_like(w).requires_grad_(True)  # every gate starts half open
    opt = torch.optim.Adam([w, alpha], lr=lr)
    history = []
    for s in range(steps):
        T_s = temp[0] * (temp[1] / temp[0]) ** (s / max(steps - 1, 1))
        gate = torch.sigmoid(alpha / T_s)
        loss = ((An @ (gate * w).T - bn) ** 2).sum() + lam * gate.sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % max(1, steps // 40) == 0:
            history.append(gate.detach().clone())
    support = torch.sigmoid(alpha.detach() / temp[1]) > 0.5
    xi = masked_ridge(An.T @ An, (An.T @ bn).T, support, 1e-12) * sb[:, None] / sa[None, :]
    return {"support": support, "xi": xi.detach(), "history": torch.stack(history)}


# ---------------------------------------------------------------- the LaTeX writer and reader
def _coef_text(v: float, e: float | None, digits: int) -> str:
    if e is not None and e > 0 and math.isfinite(e):
        dec = max(0, 1 - math.floor(math.log10(e)))  # two significant digits of the uncertainty
        return f"({abs(v):.{dec}f} \\pm {e:.{dec}f})"
    return f"{abs(v):.{digits}g}"


def write_equation(lhs: str, terms, xi: torch.Tensor, err: torch.Tensor | None = None, digits: int = 6) -> str:
    """'lhs = c_1\\, term_1 + c_2\\, term_2 …' over the non-zero coefficients, with ± uncertainties when given."""
    out = []
    for k in torch.nonzero(xi).flatten().tolist():
        v = float(xi[k])
        txt = _coef_text(v, None if err is None else float(err[k]), digits) + "\\, " + terms[k].latex
        out.append(("-" if v < 0 else "") + txt if not out else (" - " if v < 0 else " + ") + txt)
    return f"{lhs} = " + ("".join(out) if out else "0")


def read_equation(line: str, terms) -> torch.Tensor:
    """The coefficient vector ξ (K,) from a written equation alone (the inverse of write_equation)."""
    names = {tm.latex: k for k, tm in enumerate(terms)}
    xi = torch.zeros(len(terms), dtype=torch.float64)
    rest = line.split(" = ", 1)[1]
    if rest == "0":
        return xi
    pieces = re.split(r" ([+-]) ", rest)
    signs = ["-" if pieces[0].startswith("-") else "+"] + pieces[1::2]
    chunks = [pieces[0].lstrip("-")] + pieces[2::2]
    for sgn, chunk in zip(signs, chunks):
        coef, latex = chunk.split("\\, ", 1)
        value = float(coef.strip("()").split(" \\pm ")[0])
        xi[names[latex]] = -value if sgn == "-" else value
    return xi


# ---------------------------------------------------------------- re-simulating the found equation
SUBSTEPS = {"lorenz": 2, "heat": 30, "burgers": 15, "maxwell": 4}  # RK4 steps per data interval


def simulate_found(ds: dict, xi: torch.Tensor, n_times: int | None = None) -> torch.Tensor:
    """Integrate the found equation (RK4, spectral derivatives) from the first data snapshot; output at the data
    times, shape (F, *space, n_times)."""
    U0, hs = ds["U"][..., 0], ds["h"][:-1]
    n_times = n_times or ds["U"].shape[-1]
    sub = SUBSTEPS[ds["name"]]
    dt = ds["h"][-1] / sub
    U, out = U0.clone(), [U0]

    def f(V):
        return eq.found_rhs(V, hs, ds["terms"], xi)

    for _ in range((n_times - 1) * sub):
        k1 = f(U)
        k2 = f(U + 0.5 * dt * k1)
        k3 = f(U + 0.5 * dt * k2)
        U = U + dt / 6 * (k1 + 2 * k2 + 2 * k3 + f(U + dt * k3))
        out.append(U)
    return torch.stack(out[::sub], -1)


# ---------------------------------------------------------------- one discovery and the noise study
def build(ds: dict, U: torch.Tensor, method: str):
    """(Θ, ∂_t U) for a data set in the strong (fd, sg) or weak form."""
    if method == "weak":
        phi = eq.test_functions(U.shape[1:], ds["h"], ds["weak"]["radius"], ds["weak"]["stride"])
        return eq.weak_library(U, ds["h"], ds["terms"], phi)
    th, ut, _ = eq.strong_library(U, ds["h"], ds["terms"], method)
    return th, ut


def discover(ds: dict, method: str, noise: float, seed: int = 0, n_boot: int = 100, regressor: str = "stridge",
             gate_steps: int = 800) -> dict:  # fmt: skip
    """Add noise, build the library, find the sparse equation, refit with bootstrap error bars, write it in LaTeX."""
    t0 = time.perf_counter()
    U = T.add_noise(ds["U"], noise, seed + 1)
    th, ut = build(ds, U, method)
    g = torch.Generator().manual_seed(seed)
    rows = torch.randperm(len(th), generator=g)[:MAX_ROWS]
    th, ut = th[rows], ut[rows]
    n_tr = int(0.8 * len(th))
    paths = torch.stack([stridge_path(th[:n_tr], ut[:n_tr], lam=lam) for lam in LAMS])  # (L, tols, E, K)
    pick, val_err = select(paths.flatten(0, 1), th[n_tr:], ut[n_tr:])
    lam_idx, tol_idx = pick // len(TOLS), pick % len(TOLS)
    E = ut.shape[1]
    path = torch.stack([paths[lam_idx[e], :, e] for e in range(E)], 1)  # the chosen λ's path per equation (tols, E, K)
    if regressor == "gate":
        support = gate_search(th[:n_tr], ut[:n_tr], steps=gate_steps, seed=seed)["support"]
    else:
        support = torch.stack([path[tol_idx[e], e] != 0 for e in range(E)])
    xi, err = torch.zeros_like(support, dtype=th.dtype), torch.zeros_like(support, dtype=th.dtype)
    for e in range(ut.shape[1]):
        cols = torch.nonzero(support[e]).flatten()
        xi[e, cols], err[e, cols] = bootstrap(th[:, cols], ut[:, e], n_boot=n_boot, seed=seed)
    truth = ds["truth"]
    true_terms = truth != 0
    coef_err = float(((xi - truth).abs() / truth.abs())[true_terms].max())
    return {"target": ds["name"], "method": method if regressor == "stridge" else f"gate-{method}", "noise": noise,
            "terms": ds["terms"], "xi": xi, "err": err, "support": support,
            "success": bool(torch.equal(support, true_terms)), "coef_err": coef_err,
            "lines": [write_equation(lhs, ds["terms"], xi[e], err[e]) for e, lhs in enumerate(ds["lhs"])],
            "path": path, "share": path * th[:n_tr].norm(dim=0) / ut[:n_tr].norm(dim=0)[:, None], "pick": tol_idx,
            "lam": [LAMS[i] for i in lam_idx], "rows": len(th),
            "seconds": time.perf_counter() - t0}  # fmt: skip


def noise_study(levels=T.NOISE_LEVELS, methods=("fd", "sg", "weak"), small: bool = False, n_boot: int = 100,
                gate: bool = False, gate_steps: int = 800, datasets: dict | None = None, log=None) -> list:  # fmt: skip
    """Every target × noise level × derivative method (and, with gate=True, the annealed-gate regression on the weak
    form). Returns one result dict per run."""
    datasets = datasets or {n: T.dataset(n, small) for n in T.TARGETS}
    out = []
    for ds in datasets.values():
        for lv in levels:
            runs = [("stridge", m) for m in methods] + ([("gate", "weak")] if gate else [])
            for reg, m in runs:
                r = discover(ds, m, lv, n_boot=n_boot, regressor=reg, gate_steps=gate_steps)
                out.append(r)
                if log:
                    log(r)
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/equation_finder/plot.py
"""The animation: for each target, the data evolving next to a re-simulation of the equation found, the STRidge
tolerance sweep pruning the library (term shares shrinking to zero), the noise study, and the equation as it is
written. One GIF per target; a static summary of the four equations found from 1 % noisy data."""

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from problems._common.plotting import ACCENT, export_plot_data
from problems._common.progress import report
from problems._common.runtime import fast_mode

from . import solve
from . import tensors as T

METHODS = {  # colour, line style, label
    "fd": (ACCENT[1], "-", "finite differences"),
    "sg": (ACCENT[4], "-", "Savitzky–Golay"),
    "weak": (ACCENT[0], "-", "weak form"),
    "gate-weak": (ACCENT[2], "--", "annealed gates (weak)"),
}
TITLES = {
    "lorenz": "Lorenz system",
    "heat": "heat equation",
    "burgers": "viscous Burgers",
    "maxwell": "Maxwell (2D TM)",
}
SHARE_LIM = (1e-5, 3.0)
ERR_LIM = (1e-7, 300.0)
SWEEP, HOLD = 14, 6  # frames per GIF: tolerance sweep, then the final equation (last frame held 4 s)


def save_gif_shared_palette(fig, update, frames: list, path: Path, fps: int = 5, durations=None) -> Path:
    """Every frame quantised with ONE palette built from a strip of frames, so colours do not drift between frames."""
    fig.set_dpi(80)
    shots = []
    for i, f in enumerate(frames):
        update(f)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    export_plot_data(fig, path)
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for k, im in enumerate(sample):
        strip.paste(im, (0, k * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=durations or int(1000 / fps), loop=0)
    return tmp.replace(path)


def _short(line: str, n_terms: int = 6) -> str:
    """Keep the first n terms of a long written equation."""
    lhs, rest = line.split(" = ", 1)
    parts = re.split(r" (?=[+-] )", rest)
    if len(parts) <= n_terms:
        return line
    return f"{lhs} = " + " ".join(parts[:n_terms]) + f" + \\ldots\\ ({len(parts) - n_terms}\\ more)"


def _wrap(line: str, per_line: int) -> list[str]:
    """Break a written equation after every per_line terms (continuation lines start with the sign)."""
    lhs, rest = line.split(" = ", 1)
    parts = re.split(r" (?=[+-] )", rest)
    chunks = [" ".join(parts[i : i + per_line]) for i in range(0, len(parts), per_line)]
    return [f"{lhs} = {chunks[0]}"] + [f"\\qquad {c}" for c in chunks[1:]]


def constants(name: str, terms, xi) -> str:
    """The physical constants read off the found coefficients."""
    n = [tm.latex for tm in terms]
    x = xi.numpy()
    if name == "lorenz":
        return (f"$\\sigma$ = {(x[0, n.index('y')] - x[0, n.index('x')]) / 2:.4f} (10),  "
                f"$\\rho$ = {x[1, n.index('x')]:.3f} (28),  $\\beta$ = {-x[2, n.index('z')]:.4f} (8/3)")  # fmt: skip
    if name == "heat":
        return f"$D$ = {x[0, n.index('u_{xx}')]:.5f} cm$^2$/s  (true {T.HEAT_D})"
    if name == "burgers":
        return f"$\\nu$ = {x[0, n.index('u_{xx}')]:.5f} m$^2$/s  (true {T.BURGERS_NU})"
    c2 = (x[0, n.index("\\partial_x B_y")] - x[0, n.index("\\partial_y B_x")]) / 2
    return f"$c$ = {np.sqrt(max(c2, 0)):.3f} cm/ns  (true {T.C_LIGHT:.3f})"


def _data_panel(fig, spec, ds, sim):
    """Build the data panel once; returns update(k) that moves it to time index k."""
    name, U, S = ds["name"], ds["U"].numpy(), sim.numpy()
    t = ds["coords"][-1].numpy()
    if name == "maxwell":
        sub = spec.subgridspec(1, 3, width_ratios=[1, 1, 0.07], wspace=0.15)
        vmax = float(np.abs(U[0]).max())
        x = ds["coords"][0].numpy()
        ext = [x[0], x[-1], x[0], x[-1]]
        ims, axes = [], []
        for j, lab in enumerate(("data (Yee FDTD)", "found equation")):
            ax = fig.add_subplot(sub[j])
            ims.append(ax.imshow(U[0, :, :, 0].T, origin="lower", extent=ext, cmap="RdBu_r", vmin=-vmax, vmax=vmax))
            ax.set_xlabel("$x$ (cm)", fontsize=10)
            if j == 0:
                ax.set_ylabel("$y$ (cm)", fontsize=10)
            else:
                ax.set_yticklabels([])
            ax.tick_params(labelsize=10)
            axes.append((ax, lab))
        cb = fig.colorbar(ims[0], cax=fig.add_subplot(sub[2]))
        cb.set_label("$E_z$ (a.u.)", fontsize=10)
        cb.ax.tick_params(labelsize=9)

        def update(k):
            for im, A, (ax, lab) in zip(ims, (U, S), axes):
                im.set_data(A[0, :, :, k].T)
                ax.set_title(f"{lab}, t = {t[k] * 1000:.0f} ps", fontsize=11)

        return update
    ax = fig.add_subplot(spec)
    ax.tick_params(labelsize=10)
    if name == "lorenz":
        ax.plot(U[0, 0], U[2, 0], color="#d1d5db", lw=0.8)
        ld = ax.plot([], [], color=ACCENT[0], lw=1.6, label="data")[0]
        lf = ax.plot([], [], color=ACCENT[4], lw=1.4, ls="--", label="found equation")[0]
        both = np.concatenate([U[[0, 2], 0], S[[0, 2], 0]], 1)
        lo, hi = both.min(1), both.max(1)
        pad = 0.05 * (hi - lo)
        ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
        ax.set_ylim(lo[1] - pad[1], hi[1] + pad[1])
        ax.set_xlabel("$x$ (a.u.)", fontsize=10)
        ax.set_ylabel("$z$ (a.u.)", fontsize=10)
        ax.legend(fontsize=9, loc="upper right")

        def update(k):
            ld.set_data(U[0, 0, : k + 1], U[2, 0, : k + 1])
            lf.set_data(S[0, 0, : k + 1], S[2, 0, : k + 1])
            ax.set_title(f"one trajectory, $t$ = {t[k]:.2f} (a.u.)", fontsize=11)

        return update
    x = ds["coords"][0].numpy()
    unit, xu = ("K", "cm") if name == "heat" else ("m/s", "m")
    ax.plot(x, U[0, :, 0], color="#d1d5db", lw=1.0, label="$t = 0$")
    ld = ax.plot(x, U[0, :, 0], color=ACCENT[0], lw=2.0, label="data")[0]
    lf = ax.plot(x, S[0, :, 0], color=ACCENT[4], lw=1.6, ls="--", label="found equation")[0]
    lo, hi = float(U.min()), float(U.max())
    pad = 0.08 * (hi - lo)
    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel(f"$x$ ({xu})", fontsize=10)
    ax.set_ylabel(f"$u$ ({unit})", fontsize=10)
    ax.legend(fontsize=9, loc="upper right")

    def update(k):
        ld.set_ydata(U[0, :, k])
        lf.set_ydata(S[0, :, k])
        ax.set_title(f"$t$ = {t[k]:.2f} s", fontsize=11)

    return update


def _bars_panel(fig, spec, ds, r):
    """Build the bar panel once; returns update(i) that shows tolerance step i of the STRidge sweep."""
    E, K = r["share"].shape[1:]
    sub = spec.subgridspec(E, 1, hspace=0.12)
    names = [f"${tm.latex}$" for tm in ds["terms"]]
    truth = ds["truth"].numpy() != 0
    base = SHARE_LIM[0] * 0.5
    bars, top = [], None
    for e in range(E):
        ax = fig.add_subplot(sub[e])
        bars.append(ax.bar(np.arange(K), np.zeros(K), bottom=base, width=0.7, color=ACCENT[0]))
        ax.scatter(np.where(truth[e])[0], np.full(truth[e].sum(), SHARE_LIM[0] * 1.8), marker="^", color="#111827",
                   s=22, zorder=3)  # fmt: skip
        ax.set_yscale("log")
        ax.set_ylim(*SHARE_LIM)
        ax.set_xlim(-0.6, K - 0.4)
        ax.set_yticks([1e-4, 1e-2, 1])
        ax.minorticks_off()
        ax.tick_params(labelsize=9)
        if E == 1:
            ax.set_ylabel(f"share in ${ds['lhs'][e]}$ (dimensionless)", fontsize=10)
        else:  # one shared axis label for the stack, the equation named inside each row
            ax.text(0.99, 0.9, f"${ds['lhs'][e]}$", transform=ax.transAxes, ha="right", va="top", fontsize=12,
                    bbox={"fc": "white", "ec": "#9ca3af", "boxstyle": "round,pad=0.2"})  # fmt: skip
        ax.set_xticks(np.arange(K))
        ax.set_xticklabels(names if e == E - 1 else [], rotation=90, fontsize=9 if K > 14 else 10)
        top = top or ax
    if E > 1:
        box = spec.get_position(fig)
        fig.text(box.x0 - 0.042, (box.y0 + box.y1) / 2, "share (dimensionless)", rotation=90, ha="center",
                 va="center", fontsize=10)  # fmt: skip

    def update(i):
        kept = 0
        for e in range(E):
            sh = r["share"][min(i, int(r["pick"][e])), e].numpy()
            kept += int((sh != 0).sum())
            for rect, v in zip(bars[e], sh):
                rect.set_height(abs(v) - base if v != 0 else 0.0)
                rect.set_color(ACCENT[0] if v >= 0 else ACCENT[1])
        tol = float(solve.TOLS[min(i, int(r["pick"].max()))])
        top.set_title(f"term share $|\\xi_k|\\,\\|\\Theta_k\\|/\\|\\partial_t u\\|$ (dimensionless): STRidge "
                      f"$\\tau$ = {tol:.0e}, {kept} terms kept", fontsize=11)  # fmt: skip

    return update


def _noise_panel(fig, spec, rows, name: str, levels):
    ax = fig.add_subplot(spec)
    xs = np.arange(len(levels))
    for m, (col, ls, lab) in METHODS.items():
        rr = [r for r in rows if r["target"] == name and r["method"] == m]
        if not rr:
            continue
        err = np.array([max(r["coef_err"] * 100, ERR_LIM[0]) for r in rr])
        ok = np.array([r["success"] for r in rr])
        ax.plot(xs, err, color=col, ls=ls, lw=1.6, label=lab)
        ax.scatter(xs[ok], err[ok], color=col, s=40, zorder=3)
        ax.scatter(xs[~ok], err[~ok], color=col, marker="x", s=55, zorder=3)
    ax.set_yscale("log")
    ax.set_ylim(*ERR_LIM)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{lv * 100:g}" for lv in levels])
    ax.set_xlim(-0.3, len(levels) - 0.7)
    ax.set_xlabel("noise (% of the field's std)", fontsize=10)
    ax.set_ylabel("largest coefficient error (%)", fontsize=10)
    ax.set_title("noise study: dot = right terms, × = wrong terms", fontsize=11)
    ax.legend(fontsize=9, loc="lower right", ncol=2)
    ax.tick_params(labelsize=10)
    ax.grid(alpha=0.3)


def _summary(fig, datasets, rows, levels):
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    ax.text(0.5, 0.95, "Equations found from data with 1 % noise (weak form + STRidge, ± bootstrap)",
            ha="center", fontsize=15, weight="bold")  # fmt: skip
    y = 0.86
    for tgt in datasets:
        r = next((q for q in rows if q["target"] == tgt and q["method"] == "weak" and q["noise"] == 0.01),
                 None) or next(q for q in rows if q["target"] == tgt and q["method"] == "weak")  # fmt: skip
        ax.text(0.04, y, f"{TITLES[tgt]}  ({'right terms' if r['success'] else 'wrong terms'};  "
                f"{constants(tgt, r['terms'], r['xi'])})", fontsize=12, color="#374151")  # fmt: skip
        y -= 0.055
        for line in r["lines"]:
            ax.text(0.07, y, f"${_short(line, 4)}$", fontsize=12.5)
            y -= 0.05
        y -= 0.02
    verdict = []
    for m in ("fd", "sg", "weak", "gate-weak"):
        ok = [tgt for tgt in datasets if any(q["target"] == tgt and q["method"] == m and q["noise"] == max(levels)
                                             and q["success"] for q in rows)]  # fmt: skip
        verdict.append(f"{METHODS[m][2]}: {', '.join(ok) if ok else 'none'}")
    ax.text(0.04, y - 0.01, f"right terms at {max(levels) * 100:g} % noise:", fontsize=12, color=ACCENT[1])
    ax.text(0.07, y - 0.06, ";   ".join(verdict), fontsize=12, color=ACCENT[1])


def animate_target(name: str, ds: dict, rows: list[dict], r: dict, sim, path: Path) -> Path:
    """One GIF for one target: the data evolving next to the re-simulated found equation, the STRidge sweep pruning
    the library, the noise study, and (bottom right) the equation written by the finder; the final frame is held."""
    levels = sorted({q["noise"] for q in rows})
    sweep, final_frames = (3, 1) if fast_mode() else (SWEEP, HOLD)
    fig = plt.figure(figsize=(13.5, 8.0))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.25], height_ratios=[1.1, 1], left=0.06, right=0.98,
                          top=0.91, bottom=0.07, wspace=0.3, hspace=0.5)  # fmt: skip
    fig.suptitle(f"{TITLES[name]}: find $\\partial_t u = \\Theta(u)\\,\\xi$ with sparse $\\xi$ "
                 f"(clean data, finite differences)", fontsize=14, weight="bold")  # fmt: skip
    data = _data_panel(fig, gs[0, 0], ds, sim)
    bars = _bars_panel(fig, gs[0, 1], ds, r)
    _noise_panel(fig, gs[1, 0], rows, name, levels)
    ax = fig.add_subplot(gs[1, 1])
    ax.axis("off")
    n_t, n_f = ds["U"].shape[-1], sweep + final_frames
    truth = [solve.write_equation(lhs, ds["terms"], ds["truth"][e], digits=4) for e, lhs in enumerate(ds["lhs"])]

    def update(f):
        k = round(f / (n_f - 1) * (n_t - 1))
        i = round(min(f, sweep - 1) / max(sweep - 1, 1) * int(r["pick"].max()))
        data(k)
        bars(i)
        for txt in list(ax.texts):
            txt.remove()
        final = f >= sweep
        ax.set_title("the equation found" + (" (final, ± bootstrap)" if final else " (pruning…)"), fontsize=12,
                     weight="bold" if final else "normal")  # fmt: skip
        if final:
            lines = r["lines"]
        else:
            lines = [solve.write_equation(lhs, ds["terms"], r["path"][min(i, int(r["pick"][e])), e], digits=3)
                     for e, lhs in enumerate(ds["lhs"])]  # fmt: skip
        y = 0.98
        for line in lines:
            for part in _wrap(_short(line, 5), 2 if final else 3):
                ax.text(0.0, y, f"${part}$", fontsize=12.5, va="top", transform=ax.transAxes)
                y -= 0.11
            y -= 0.02
        if final:
            ax.text(0.0, y, constants(name, ds["terms"], r["xi"]), fontsize=12, va="top", color=ACCENT[2],
                    transform=ax.transAxes)  # fmt: skip
            y -= 0.12
            ax.text(0.0, y, "true equation:", fontsize=10.5, va="top", color="#6b7280", transform=ax.transAxes)
            for t in truth:
                y -= 0.085
                ax.text(0.03, y, f"${t}$", fontsize=10.5, va="top", color="#6b7280", transform=ax.transAxes)

    durations = [200] * (n_f - 1) + [4000]  # the finished equation stays 4 s before the loop
    return save_gif_shared_palette(fig, update, list(range(n_f)), path, durations=durations)


def summary_png(datasets: dict, rows: list[dict], path: Path) -> Path:
    """Static summary: the four equations found from 1 % noisy data, and who survives 5 % noise."""
    fig = plt.figure(figsize=(13.5, 7.6), dpi=80)
    _summary(fig, datasets, rows, sorted({q["noise"] for q in rows}))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=80)
    plt.close(fig)
    return path


def animate(datasets: dict, rows: list[dict], show: dict, sims: dict, out_dir: Path, main: str = "maxwell") -> Path:
    """results/<target>.gif for every target except the main one, which is the problem's main GIF; plus
    results/summary.png."""
    res = out_dir / "results"
    res.mkdir(parents=True, exist_ok=True)
    main_path = out_dir / "equation_finder.gif"
    for name in datasets:
        path = main_path if name == main else res / f"{name}.gif"
        animate_target(name, datasets[name], rows, show[name], sims[name], path)
    summary_png(datasets, rows, res / "summary.png")
    return main_path


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'bootstrap': 100,
    'gate_steps': 800,
    'maxwell_grid': 128
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.equation_finder")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.equation_finder.main")
problem_dir = Path("problems/ai/equation_finder")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
